# Distributions

Target: `OD_PAX`, passengers on one origin-destination route in one departure month.

Plots and the skew numbers use a 50,000-row sample (`random_state=9`). The row count uses the full forecasting file.

This notebook is the distribution pass. The feature set it points to is `build_features` in `pri-biswal.ipynb`: gradient-boosted trees, with same-month revenue left out.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists())
df_pax = pd.read_csv(root / "data" / "raw" / "AA_PAX_FORECASTING.csv")

print(f"Full row count: {len(df_pax):,}")
assert len(df_pax) == 1_533_522

df_sample = df_pax.sample(n=50_000, random_state=9).copy()
print(f"Sample shape: {df_sample.shape}")


In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(data=df_sample, x="OD_PAX", bins=80, element="step", color="royalblue")
plt.title("Histogram of passenger volume (OD_PAX)")
plt.xlabel("OD_PAX")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
sns.histplot(data=df_sample, x="OD_PAX", bins=80, element="step", color="teal")
plt.xscale("log")
plt.title("Histogram of passenger volume (OD_PAX) on a log scale")
plt.xlabel("OD_PAX")
plt.tight_layout()
plt.show()

raw_skew = df_sample["OD_PAX"].skew()
log_skew = np.log1p(df_sample["OD_PAX"]).skew()
print(f"Raw OD_PAX skewness: {raw_skew:.4f}")
print(f"Log-transformed OD_PAX skewness: {log_skew:.4f}")

summary_stats = df_sample.groupby("DEP_MONTH")["OD_PAX"].agg(["count", "mean", "median", "std"])
display(summary_stats.round(1))


The raw plot is extremely right-skewed (skew 5.3348 on this sample). Most route-months have a low passenger count, and a few busy ones spike. The log scale pulls that tail in (skew -0.1311).

In every month the mean is several times the median. January's mean is about 2,447 and its median is about 432. A handful of busy routes dominate the average.

Squared error spends the fit on those large absolute misses. Gradient-boosted trees still fit this shape, and Deliverable 4 should tune them with absolute error or Huber rather than squared error.


In [ ]:
numeric_cols = df_sample.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(8, 5))
sns.heatmap(df_sample[numeric_cols].corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation of numeric columns, 50k sample")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.regplot(
    data=df_sample,
    x="DEP_MONTH",
    y="OD_PAX",
    scatter_kws={"alpha": 0.1, "color": "royalblue"},
    line_kws={"color": "red"},
)
plt.title("OD_PAX against departure month, with a straight-line fit")
plt.xlabel("Departure month")
plt.ylabel("OD_PAX")
plt.tight_layout()
plt.show()

print(
    "Sample correlation, OD_REV_USD with OD_PAX:",
    round(df_sample["OD_REV_USD"].corr(df_sample["OD_PAX"]), 3),
)


## What this changes in the model

The forecasting file is tall and narrow: 1,533,522 rows and 8 columns. The route and the month are in the row. Airport attributes have to be joined on. That join is `build_features`.

The forecasting file has no missing cells. Blank state and country codes are in the airport lookup. A blank state is a non-US airport, not a route that failed to run and not a passenger count of zero. `STATE_PROVNC_CD` stays, filled with `MISSING`.

`OD_REV_USD` correlates about 0.76 with `OD_PAX` on this sample. It is the revenue from those same passengers in that same month. Using it would score the model on something that is not known at the start of the month. It is dropped. The correlation is the reason it looks useful, and the reason it cannot be a feature.

The file is many routes stacked by month. It is not one continuous timeline, so ARIMA on a single series does not price a route-month.

Gradient-boosted trees are the model. They take the route categories and the seasonal month without treating `DEP_MONTH` as one straight line through every route.
